In [8]:
import sys
import pandas as pd

print(sys.executable)
print(sys.version)
print(pd.__version__)

e:\GenAiClass\generative-ai-pgp-ji-2026\.venv\Scripts\python.exe
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
2.2.3


In [9]:
df = pd.read_csv("bbc-news-data.csv", sep="\t")

In [10]:
print(df.shape)
df.head()

(2225, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [11]:
df = pd.read_csv("bbc-news-data.csv", sep="\t")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Dataset shape: (2225, 4)
Columns: ['category', 'filename', 'title', 'content']


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [12]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2225 entries, 0 to 2224
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   category  2225 non-null   object
 1   filename  2225 non-null   object
 2   title     2225 non-null   object
 3   content   2225 non-null   object
dtypes: object(4)
memory usage: 69.7+ KB
None


In [13]:
df.isnull().sum()

category    0
filename    0
title       0
content     0
dtype: int64

In [14]:
df_30 = df.head(30).copy()

print("Selected data shape:", df_30.shape)
df_30.head()

Selected data shape: (30, 4)


,category,filename,title,content
0,business,001.txt,Ad sales boost Time Warner profit,Quarterly profits at US media giant TimeWarne...
1,business,002.txt,Dollar gains on Greenspan speech,The dollar has hit its highest level against ...
2,business,003.txt,Yukos unit buyer faces loan claim,The owners of embattled Russian oil giant Yuk...
3,business,004.txt,High fuel prices hit BA's profits,British Airways has blamed high fuel prices f...
4,business,005.txt,Pernod takeover talk lifts Domecq,Shares in UK drinks and food firm Allied Dome...


In [15]:
print("TITLE:")
print(df_30.iloc[0]["title"])

print("\nCATEGORY:")
print(df_30.iloc[0]["category"])

print("\nARTICLE:")
print(df_30.iloc[0]["content"])

TITLE:
Ad sales boost Time Warner profit

CATEGORY:
business

ARTICLE:
 Quarterly profits at US media giant TimeWarner jumped 76% to $1.13bn (£600m) for the three months to December, from $639m year-earlier.  The firm, which is now one of the biggest investors in Google, benefited from sales of high-speed internet connections and higher advert sales. TimeWarner said fourth quarter sales rose 2% to $11.1bn from $10.9bn. Its profits were buoyed by one-off gains which offset a profit dip at Warner Bros, and less users for AOL.  Time Warner said on Friday that it now owns 8% of search-engine Google. But its own internet business, AOL, had has mixed fortunes. It lost 464,000 subscribers in the fourth quarter profits were lower than in the preceding three quarters. However, the company said AOL's underlying profit before exceptional items rose 8% on the back of stronger internet advertising revenues. It hopes to increase subscribers by offering the online service free to TimeWarner internet 

In [16]:
import os

print("API key loaded:", bool(os.getenv("GROQ_API_KEY")))


API key loaded: True


In [17]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

In [22]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [23]:
classification_prompt = ChatPromptTemplate.from_template("""
You are a news article classification system.

Classify the following news article into exactly ONE of these categories:

Business
Entertainment
Politics
Sport
Tech

Rules:
- Return only the category name.
- Do not provide an explanation.
- Do not return any other text.

News Article:
{article}
""")

In [24]:
classification_chain = classification_prompt | llm

In [25]:
sample_article = df_30.iloc[0]["content"]

response = classification_chain.invoke({
    "article": sample_article
})

print(response.content)

Business


In [ ]:
summarization_prompt = ChatPromptTemplate.from_template("""
You are a news article summarization system.

Summarize the following news article in 2-3 concise sentences.

Rules:
- Capture the main points of the article.
- Include important information such as who, what, when, where, or why when relevant.
- Do not add personal opinions or commentary.
- Return only the summary.

News Article:
{article}
""")

SyntaxError: invalid syntax (53701719.py, line 14)

In [28]:
summarization_chain = summarization_prompt | llm

In [29]:
summary_response = summarization_chain.invoke({
    "article": sample_article
})

print(summary_response.content)

TimeWarner reported a 76% jump in fourth‑quarter profit to $1.13 billion, driven by higher high‑speed internet sales, advertising revenue and an 8% stake in Google, while overall sales rose 2% to $11.1 billion. The company’s film division saw profits fall 27% after box‑office flops, and AOL lost subscribers despite an 8% rise in underlying profit, prompting plans to offer free AOL service to TimeWarner internet customers. TimeWarner also announced it will restate its 2000 and 2003 results and settle a $300 million SEC probe related to AOL, while projecting about 5% operating‑earnings growth for 2005.


In [30]:
entity_prompt = ChatPromptTemplate.from_template("""
You are an information extraction system.

Extract the important entities mentioned in the following news article.

Return the result in exactly this format:

People: [list of people]
Organizations: [list of organizations]
Locations: [list of locations]

Rules:
- Include only important entities relevant to the article.
- Do not invent entities.
- If no entity of a particular type is mentioned, use an empty list [].
- Return only the requested information.

News Article:
{article}
""")

In [31]:
entity_chain = entity_prompt | llm

In [32]:
entity_response = entity_chain.invoke({
    "article": sample_article
})

print(entity_response.content)

People: [Richard Parsons]  
Organizations: [TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission, Bertelsmann, AOL Europe]  
Locations: [United States, Germany]


In [33]:
def analyze_article(article):
    
    # 1. Topic classification
    topic_response = classification_chain.invoke({
        "article": article
    })
    
    # 2. Summarization
    summary_response = summarization_chain.invoke({
        "article": article
    })
    
    # 3. Entity extraction
    entity_response = entity_chain.invoke({
        "article": article
    })
    
    return {
        "Detected_Topic": topic_response.content.strip(),
        "Summary": summary_response.content.strip(),
        "Key_Entities": entity_response.content.strip()
    }

In [34]:
test_result = analyze_article(sample_article)

test_result

{'Detected_Topic': 'Business',
 'Summary': 'TimeWarner’s quarterly profit surged 76% to $1.13\u202fbillion for the three months to December, driven by higher high‑speed internet sales, advertising revenue and an 8% stake in Google, while overall sales rose 2% to $11.1\u202fbillion. The company’s film division saw a 27% profit drop, AOL lost 464,000 subscribers though its underlying profit rose 8%, and TimeWarner must restate its 2000‑2003 results after an SEC probe, offering $300\u202fmillion to settle related charges. For the full year, profit climbed 27% to $3.36\u202fbillion, and the firm projects about 5% operating‑earnings growth in 2005.',
 'Key_Entities': 'People: [Richard Parsons]\nOrganizations: [TimeWarner, Google, Warner Bros, AOL, US Securities Exchange Commission, Bertelsmann]\nLocations: [United States, Germany]'}

In [35]:
results = []

for index, row in df_30.iterrows():
    print(f"Processing article {index + 1}/30...")
    
    result = analyze_article(row["content"])
    results.append(result)

Processing article 1/30...
Processing article 2/30...
Processing article 3/30...
Processing article 4/30...


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-120b` in organization `org_01kx37kv5ze0nb30mqjgs25g0w` service tier `on_demand` on tokens per minute (TPM): Limit 8000, Used 6904, Requested 1118. Please try again in 164.999999ms. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

In [37]:
from langchain_core.output_parsers import StrOutputParser

In [38]:
combined_prompt = ChatPromptTemplate.from_template("""
You are a news article analysis system.

Analyze the following BBC news article and perform THREE tasks.

TASK 1 — Topic Classification
Classify the article into exactly ONE category:
Business, Entertainment, Politics, Sport, Tech

TASK 2 — Summarization
Write a concise summary in 2-3 sentences.
Capture the main points and important who/what/when/where/why information.
Do not add opinions.

TASK 3 — Key Entity Extraction
Identify important:
- People
- Organizations
- Locations

Return your answer ONLY in this format:

Topic: <one category>
Summary: <2-3 sentence summary>
People: <comma-separated names, or None>
Organizations: <comma-separated organizations, or None>
Locations: <comma-separated locations, or None>

Do not add any explanation outside this format.

ARTICLE:
{article}
""")

In [39]:
combined_chain = combined_prompt | llm | StrOutputParser()

In [40]:
test_output = combined_chain.invoke({
    "article": sample_article
})

print(test_output)

Topic: Business
Summary: TimeWarner reported a 76% jump in fourth‑quarter profit to $1.13 billion, helped by high‑speed internet sales and stronger advertising, while its film division saw a 27% profit decline. The media group now owns an 8% stake in Google, is facing an SEC investigation that will require restating past results, and plans to boost AOL subscriptions by offering the service free to its internet customers.
People: Richard Parsons
Organizations: TimeWarner, Google, AOL, Warner Bros, SEC, Bertelsmann
Locations: United States


In [41]:
def parse_analysis(output):
    result = {
        "Detected_Topic": "",
        "Summary": "",
        "People": "",
        "Organizations": "",
        "Locations": ""
    }

    for line in output.splitlines():
        line = line.strip()

        if line.startswith("Topic:"):
            result["Detected_Topic"] = line.replace("Topic:", "", 1).strip()

        elif line.startswith("Summary:"):
            result["Summary"] = line.replace("Summary:", "", 1).strip()

        elif line.startswith("People:"):
            result["People"] = line.replace("People:", "", 1).strip()

        elif line.startswith("Organizations:"):
            result["Organizations"] = line.replace("Organizations:", "", 1).strip()

        elif line.startswith("Locations:"):
            result["Locations"] = line.replace("Locations:", "", 1).strip()

    return result

In [43]:
def analyze_article(article):
    output = combined_chain.invoke({
        "article": article
    })

    return parse_analysis(output)

In [44]:
test_result = analyze_article(sample_article)

test_result

{'Detected_Topic': 'Business',
 'Summary': 'TimeWarner reported a 76% jump in fourth‑quarter profit to $1.13\u202fbillion, helped by high‑speed internet sales and stronger advertising, while its film division saw a 27% profit decline due to box‑office failures. The firm now owns an 8% stake in Google, is restating 2000 and 2003 results after an SEC investigation, and plans to offer free AOL service to its internet customers to boost subscriptions.',
 'People': 'Richard Parsons',
 'Organizations': 'TimeWarner, Google, AOL, Warner Bros, SEC, Bertelsmann',
 'Locations': 'United States'}

In [45]:
import time

results = []

for index, row in df_30.iterrows():
    print(f"Processing article {index + 1}/30...")

    try:
        result = analyze_article(row["content"])
        results.append(result)

        # Small delay between API calls
        time.sleep(2)

    except Exception as e:
        print(f"Error on article {index + 1}: {e}")

        results.append({
            "Detected_Topic": "ERROR",
            "Summary": "ERROR",
            "People": "ERROR",
            "Organizations": "ERROR",
            "Locations": "ERROR"
        })

        time.sleep(5)

Processing article 1/30...
Processing article 2/30...
Processing article 3/30...
Processing article 4/30...
Processing article 5/30...
Processing article 6/30...
Processing article 7/30...
Processing article 8/30...
Processing article 9/30...
Processing article 10/30...
Processing article 11/30...
Processing article 12/30...
Processing article 13/30...
Processing article 14/30...
Processing article 15/30...
Processing article 16/30...
Processing article 17/30...
Processing article 18/30...
Processing article 19/30...
Processing article 20/30...
Processing article 21/30...
Processing article 22/30...
Processing article 23/30...
Processing article 24/30...
Processing article 25/30...
Processing article 26/30...
Processing article 27/30...
Processing article 28/30...
Processing article 29/30...
Processing article 30/30...


In [46]:
results_df = pd.DataFrame(results)

print(results_df.shape)
results_df.head()

(30, 5)


,Detected_Topic,Summary,People,Organizations,Locations
0,Business,TimeWarner reported a 76% jump in fourth‑quart...,Richard Parsons,"TimeWarner, Google, AOL, Warner Bros, SEC, Ber...",United States
1,Business,The dollar rose to its highest level against t...,"Alan Greenspan, Robert Sinche","Federal Reserve, Bank of America, G7, White House","New York, London, China"
2,Business,"Menatep Group, the owner of the former Yukos p...","Jamie Firestone, Tim Osborne, Mikhail Khodorko...","Yukos, Menatep Group, Rosneft, Yugansk","Russia, United States"
3,Business,British Airways reported a 40% fall in pre‑tax...,"Rod Eddington, Martin Broughton, Mike Powell, ...","British Airways, Dresdner Kleinwort Wasserstei...","United Kingdom, United States"
4,Business,Shares in UK drinks and food group Allied Dome...,None,"Allied Domecq, Pernod Ricard, Wall Street Jour...","United Kingdom, France, London, Paris, Scotland"


In [47]:
final_df = pd.concat(
    [
        df_30.reset_index(drop=True),
        results_df.reset_index(drop=True)
    ],
    axis=1
)

print("Final shape:", final_df.shape)

Final shape: (30, 9)


In [48]:
final_df[
    [
        "title",
        "category",
        "Detected_Topic",
        "Summary",
        "People",
        "Organizations",
        "Locations"
    ]
].head(10)

,title,category,Detected_Topic,Summary,People,Organizations,Locations
0,Ad sales boost Time Warner profit,business,Business,TimeWarner reported a 76% jump in fourth‑quart...,Richard Parsons,"TimeWarner, Google, AOL, Warner Bros, SEC, Ber...",United States
1,Dollar gains on Greenspan speech,business,Business,The dollar rose to its highest level against t...,"Alan Greenspan, Robert Sinche","Federal Reserve, Bank of America, G7, White House","New York, London, China"
2,Yukos unit buyer faces loan claim,business,Business,"Menatep Group, the owner of the former Yukos p...","Jamie Firestone, Tim Osborne, Mikhail Khodorko...","Yukos, Menatep Group, Rosneft, Yugansk","Russia, United States"
3,High fuel prices hit BA's profits,business,Business,British Airways reported a 40% fall in pre‑tax...,"Rod Eddington, Martin Broughton, Mike Powell, ...","British Airways, Dresdner Kleinwort Wasserstei...","United Kingdom, United States"
4,Pernod takeover talk lifts Domecq,business,Business,Shares in UK drinks and food group Allied Dome...,None,"Allied Domecq, Pernod Ricard, Wall Street Jour...","United Kingdom, France, London, Paris, Scotland"
5,Japan narrowly escapes recession,business,Business,Japan's economy grew only 0.1% in the third qu...,"Heizo Takenaka, Paul Sheard","Japanese government, Lehman Brothers","Japan, Tokyo"
6,Jobs growth still slow in the US,business,Business,"US firms added 146,000 jobs in January, fallin...","President Bush, Rick Egelton, Ken Mayland","Labor Department, BMO Financial Group, ClearVi...",United States
7,India calls for fair trade rules,business,Business,India's finance minister Palaniappan Chidambar...,"Palaniappan Chidambaram, Gordon Brown","G7, G20, United Nations, World Bank, Internati...","London, India, China, Brazil, South Africa, Ru..."
8,Ethiopia's crop production up 24%,business,Business,Ethiopia's crop output rose to 14.27 million t...,Henri Josserand,"Food and Agriculture Organisation, World Food ...",Ethiopia
9,Court rejects $280bn tobacco case,business,Business,A US appeals court dismissed a 1999 Clinton ad...,None,"US government, Clinton administration, Altria ...","Washington, District of Columbia"


In [49]:
print(results_df.shape)

(30, 5)


In [50]:
final_df = pd.concat(
    [
        df_30.reset_index(drop=True),
        results_df.reset_index(drop=True)
    ],
    axis=1
)

print("Final shape:", final_df.shape)

Final shape: (30, 9)


In [51]:
final_df[
    [
        "title",
        "category",
        "Detected_Topic",
        "Summary",
        "People",
        "Organizations",
        "Locations"
    ]
].head(10)

,title,category,Detected_Topic,Summary,People,Organizations,Locations
0,Ad sales boost Time Warner profit,business,Business,TimeWarner reported a 76% jump in fourth‑quart...,Richard Parsons,"TimeWarner, Google, AOL, Warner Bros, SEC, Ber...",United States
1,Dollar gains on Greenspan speech,business,Business,The dollar rose to its highest level against t...,"Alan Greenspan, Robert Sinche","Federal Reserve, Bank of America, G7, White House","New York, London, China"
2,Yukos unit buyer faces loan claim,business,Business,"Menatep Group, the owner of the former Yukos p...","Jamie Firestone, Tim Osborne, Mikhail Khodorko...","Yukos, Menatep Group, Rosneft, Yugansk","Russia, United States"
3,High fuel prices hit BA's profits,business,Business,British Airways reported a 40% fall in pre‑tax...,"Rod Eddington, Martin Broughton, Mike Powell, ...","British Airways, Dresdner Kleinwort Wasserstei...","United Kingdom, United States"
4,Pernod takeover talk lifts Domecq,business,Business,Shares in UK drinks and food group Allied Dome...,None,"Allied Domecq, Pernod Ricard, Wall Street Jour...","United Kingdom, France, London, Paris, Scotland"
5,Japan narrowly escapes recession,business,Business,Japan's economy grew only 0.1% in the third qu...,"Heizo Takenaka, Paul Sheard","Japanese government, Lehman Brothers","Japan, Tokyo"
6,Jobs growth still slow in the US,business,Business,"US firms added 146,000 jobs in January, fallin...","President Bush, Rick Egelton, Ken Mayland","Labor Department, BMO Financial Group, ClearVi...",United States
7,India calls for fair trade rules,business,Business,India's finance minister Palaniappan Chidambar...,"Palaniappan Chidambaram, Gordon Brown","G7, G20, United Nations, World Bank, Internati...","London, India, China, Brazil, South Africa, Ru..."
8,Ethiopia's crop production up 24%,business,Business,Ethiopia's crop output rose to 14.27 million t...,Henri Josserand,"Food and Agriculture Organisation, World Food ...",Ethiopia
9,Court rejects $280bn tobacco case,business,Business,A US appeals court dismissed a 1999 Clinton ad...,None,"US government, Clinton administration, Altria ...","Washington, District of Columbia"


In [52]:
final_df["Actual"] = final_df["category"].str.lower()
final_df["Predicted"] = final_df["Detected_Topic"].str.lower()

accuracy = (
    final_df["Actual"] == final_df["Predicted"]
).mean()

print("Topic Classification Accuracy:", accuracy)

Topic Classification Accuracy: 0.9333333333333333


In [53]:
print("Part 1 final shape:", final_df.shape)

Part 1 final shape: (30, 11)


In [54]:
final_df.to_csv("part1_bbc_news_results.csv", index=False)

print("Part 1 results saved successfully!")

Part 1 results saved successfully!


## Conclusion

In Part 1, a LangChain-based Generative AI pipeline was developed to analyze 30 BBC News articles. The pipeline performs topic classification, 2–3 sentence summarization, and key entity extraction for people, organizations, and locations.

The generated outputs were added to the original dataset and saved as `part1_bbc_news_results.csv`.

Bonus


In [55]:

import os
import time
import pandas as pd

# Load the complete BBC dataset
bbc_all = pd.read_csv("bbc-news-data.csv", sep="\t")

print("Total articles:", len(bbc_all))

Total articles: 2225


In [56]:
checkpoint_file = "bonus_part1_progress.csv"

if os.path.exists(checkpoint_file):
    saved_results = pd.read_csv(checkpoint_file)
    print("Existing checkpoint found:", len(saved_results), "articles")
else:
    saved_results = pd.DataFrame(columns=[
        "Detected_Topic",
        "Summary",
        "People",
        "Organizations",
        "Locations"
    ])
    print("Starting Bonus Part 1 from scratch.")

Starting Bonus Part 1 from scratch.


In [57]:
start_index = len(saved_results)

print(f"Starting from article {start_index + 1} of {len(bbc_all)}")

for index in range(start_index, len(bbc_all)):

    row = bbc_all.iloc[index]

    print(f"Processing article {index + 1}/{len(bbc_all)}...")

    try:
        result = analyze_article(row["content"])
        saved_results = pd.concat(
            [saved_results, pd.DataFrame([result])],
            ignore_index=True
        )

        # Save checkpoint every 10 articles
        if (index + 1) % 10 == 0:
            saved_results.to_csv(checkpoint_file, index=False)
            print(f"Checkpoint saved: {index + 1} articles")

        time.sleep(2)

    except Exception as e:
        print(f"Error at article {index + 1}: {e}")

        # Save what we have so far
        saved_results.to_csv(checkpoint_file, index=False)

        print("Progress saved. Stopping so we don't lose data.")
        break

# Final checkpoint
saved_results.to_csv(checkpoint_file, index=False)

print("Current completed articles:", len(saved_results))

Starting from article 1 of 2225
Processing article 1/2225...
Processing article 2/2225...
Processing article 3/2225...
Processing article 4/2225...
Processing article 5/2225...
Processing article 6/2225...
Processing article 7/2225...
Processing article 8/2225...
Processing article 9/2225...
Processing article 10/2225...
Checkpoint saved: 10 articles
Processing article 11/2225...
Processing article 12/2225...
Processing article 13/2225...
Processing article 14/2225...
Processing article 15/2225...
Processing article 16/2225...
Processing article 17/2225...
Processing article 18/2225...
Processing article 19/2225...
Processing article 20/2225...
Checkpoint saved: 20 articles
Processing article 21/2225...
Processing article 22/2225...
Processing article 23/2225...
Processing article 24/2225...
Processing article 25/2225...
Processing article 26/2225...
Processing article 27/2225...
Error at article 27: Error code: 429 - {'error': {'message': 'Rate limit reached for model `openai/gpt-oss-

In [58]:
import pandas as pd
import os

checkpoint_file = "bonus_part1_progress.csv"

print("Checkpoint exists:", os.path.exists(checkpoint_file))

if os.path.exists(checkpoint_file):
    progress = pd.read_csv(checkpoint_file)
    print("Completed articles:", len(progress))

Checkpoint exists: True
Completed articles: 26


In [59]:
import platform
import os

print("CPU:", platform.processor())
print("Machine:", platform.machine())

CPU: Intel64 Family 6 Model 140 Stepping 1, GenuineIntel
Machine: AMD64


In [61]:
import subprocess

result = subprocess.run(
    ["powershell", "-Command", "(Get-CimInstance Win32_Processor).Name"],
    capture_output=True,
    text=True
)

print(result.stdout)

11th Gen Intel(R) Core(TM) i3-1115G4 @ 3.00GHz

